In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/elifsudayi/food-recipe-finder.git
%cd food-recipe-finder
!pip install -q -r requirements.txt
!mkdir -p corpus

Mounted at /content/drive
Cloning into 'food-recipe-finder'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 51 (delta 19), reused 42 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 65.28 KiB | 9.33 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/food-recipe-finder
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import sys; sys.path.append(".")
import pandas as pd
from src.retrieval.corpus import build_corpus, DIET_TAGS

raw = pd.read_csv("/content/drive/MyDrive/swe015/RAW_recipes.csv")
print(raw.shape, list(raw.columns))

corpus = build_corpus(raw, per_class=30)
print(len(corpus))
print(corpus["dish"].value_counts())

(231637, 12) ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']
300
dish
pizza                  30
sushi                  30
hamburger              30
ice_cream              30
caesar_salad           30
spaghetti_carbonara    30
pancakes               30
omelette               30
french_fries           30
chicken_curry          30
Name: count, dtype: int64


In [ ]:
print(corpus["minutes"].describe(), "\n")
diet = corpus["diet_tags"].str.split("|").explode()
print("distribution of diet labels:\n", diet[diet != ""].value_counts(), "\n")
print("which DIET_TAGS are completely absent from the data:", [t for t in DIET_TAGS if t not in set(diet)])
print("blank explanation rate:", (corpus["description"] == "").mean())

for dish in corpus["dish"].unique():
    print(f"\n[{dish}]", corpus[corpus.dish == dish]["title"].head(6).tolist())

count    300.000000
mean      42.726667
std       52.861388
min        5.000000
25%       20.000000
50%       30.000000
75%       45.000000
max      420.000000
Name: minutes, dtype: float64 

distribution of diet labels:
 diet_tags
low-sodium     59
vegetarian     45
low-carb       45
low-calorie    36
low-fat        22
vegan          12
gluten-free     6
Name: count, dtype: int64 

which DIET_TAGS are completely absent from the data: ['dairy-free']
blank explanation rate: 0.0

[pizza] ['Weight Watchers Pita Pizza', 'Quick Tortilla Pizza', 'Simply Delicious Shrimp And Spinach Pizza', 'Ham And Pineapple Pizza Buns', 'Portabella Pizza With Peppery Greens', 'Heather S Fruit Pizza Quick And Simple']

[sushi] ['California Roll Sushi', 'Sushi Sandwich', 'Philly Roll Sushi', 'Egg Filling For Sushi', 'Sushi Style Roll Ups', 'Spicy Tuna Roll Sushi']

[hamburger] ['Tsr Version Of Burger King Whopper By Todd Wilbur', 'Bacon Lovers Burger', 'Hamburger Yum Yum', 'Jamaican Jerk Burgers For Grill', '

In [ ]:
from src.retrieval.queries import build_queries
queries = build_queries(corpus)
print(len(queries), queries["type"].value_counts().to_dict())
print(queries[["query_id", "type", "query", "n_relevant", "criteria"]].to_string(max_colwidth=55))

40 {'ingredient': 10, 'semantic': 10, 'time': 8, 'diet': 8, 'combo': 4}
   query_id        type                                                   query  n_relevant                                                criteria
0       q01        time                            quick pizza under 20 minutes           9                                 dish=pizza & minutes<20
1       q02        time                             sushi in 30 minutes or less          17                                dish=sushi & minutes<=30
2       q03        time                   hamburger that takes less than 45 min          21                             dish=hamburger & minutes<45
3       q04        time                        easy ice cream within 60 minutes          19                            dish=ice_cream & minutes<=60
4       q05        time                     quick caesar salad under 20 minutes          15                          dish=caesar_salad & minutes<20
5       q06        time               sp

In [ ]:
import random
random.seed(7)
by_id = corpus.set_index("recipe_id")
for qid in random.sample(list(queries["query_id"]), 8):
    q = queries[queries.query_id == qid].iloc[0]
    ids = [int(i) for i in q["relevant_ids"].split("|")][:4]
    print(f"\n{qid} [{q['type']}] {q['query']!r}  ({q['n_relevant']} relevant)  | {q['criteria']}")
    for i in ids:
        r = by_id.loc[i]
        print(f"   - {r['title']} | {r['minutes']} minutes | {r['diet_tags']} | {r['ingredients'][:70]}")


q21 [ingredient] 'caesar salad with chicken'  (8 relevant)  | dish=caesar_salad & ingredients match /\bchicken\b/
   - Chicken Caesar Salad For Two | 20 minutes |  | boneless skinless chicken breast halves; olive oil; dried basil; dried
   - Grilled Chicken Caesar Salad | 15 minutes |  | mixed salad greens; grilled chicken breast strips; seasoned croutons; 
   - Springtime Chicken Tossed Caesar Salad | 15 minutes | low-calorie|low-carb|low-fat|low-sodium | romaine lettuce; cooked boneless skinless chicken breasts; cherry toma
   - Grilled Chicken Caesar Salad A New Way | 35 minutes | low-carb|low-sodium | boneless skinless chicken breast; olive oil; salt and pepper; romaine 

q10 [diet] 'vegetarian pizza'  (5 relevant)  | dish=pizza & diet_tag=vegetarian
   - Heather S Fruit Pizza Quick And Simple | 25 minutes | vegetarian | pillsbury sugar cookie dough; confectioners' sugar; vanilla; cream che
   - Green Tomatoes Zucchini Pizza My Way To Have Fried Green Tomatoes | 55 minutes | veget

In [ ]:
corpus.to_csv("corpus/recipes.csv", index=False)
queries.to_csv("corpus/queries.csv", index=False)
print(pd.read_csv("corpus/recipes.csv").shape, pd.read_csv("corpus/queries.csv").shape)

!cp corpus/recipes.csv corpus/queries.csv /content/drive/MyDrive/swe015/

(300, 8) (40, 6)
